# Visual Genome EDA: Full vs Sample 10k

Notebook này so sánh toàn bộ Visual Genome với sample 10k hiện tại theo đúng trọng tâm:
- Mức độ bao phủ label cho object, attribute, relation
- Phân phối bbox theo width, height, area, aspect ratio
- Chất lượng dữ liệu và các điểm lệch quan trọng giữa full và sample

Lưu ý:
- Sample 10k ở đây là sample image_id thực tế mà pipeline đang dùng, nên số lượng có thể lệch nhẹ so với 10,000 sau khi lọc preprocessing.
- Phần split train/val/test được bỏ khỏi EDA vì pipeline hiện đã dùng logic sample riêng cho từng task.

In [1]:
import gc
import json
import sys
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display


def resolve_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs" / "config.yaml").exists() and (candidate / "data" / "raw" / "objects.json").exists():
            return candidate
    return current


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessing import _normalize_object_name, _normalize_relation_name

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OBJECT_PROCESSED_DIR = PROCESSED_DIR / "object"
ATTRIBUTE_PROCESSED_DIR = PROCESSED_DIR / "attribute"
RELATION_PROCESSED_DIR = PROCESSED_DIR / "relation"

TOP_N = 20
LONG_TAIL_THRESHOLD = 20
HEATMAP_BINS = 60


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def extract_records(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        if "samples" in payload:
            return payload["samples"]
        if "images" in payload:
            return payload["images"]
    raise TypeError(f"Unsupported JSON payload type: {type(payload)!r}")


def load_processed_image_ids(processed_dir: Path) -> Tuple[set[int], Dict[str, int]]:
    image_ids: set[int] = set()
    split_sizes: Dict[str, int] = {}
    for split in ("train", "val", "test"):
        path = processed_dir / split / "annotations.json"
        if not path.exists():
            continue
        records = extract_records(load_json(path))
        split_sizes[split] = len(records)
        image_ids.update(int(record["image_id"]) for record in records if "image_id" in record)
    return image_ids, split_sizes


def load_image_size_map(image_data_path: Path) -> Dict[int, Tuple[float, float]]:
    records = extract_records(load_json(image_data_path))
    image_sizes: Dict[int, Tuple[float, float]] = {}
    for record in records:
        image_id = record.get("image_id")
        width = record.get("width")
        height = record.get("height")
        if image_id is None or width is None or height is None:
            continue
        image_sizes[int(image_id)] = (float(width), float(height))
    return image_sizes


def sorted_frequencies(counter: Counter) -> np.ndarray:
    values = np.array(sorted(counter.values(), reverse=True), dtype=float)
    if values.size == 0:
        return values
    return values / values.sum()


def label_coverage_summary(full_counter: Counter, sample_counter: Counter, sample_name: str) -> pd.DataFrame:
    full_labels = set(full_counter)
    sample_labels = set(sample_counter)
    shared_labels = full_labels & sample_labels
    only_full = full_labels - sample_labels
    only_sample = sample_labels - full_labels
    full_total = sum(full_counter.values())
    sample_total = sum(sample_counter.values())
    full_tail = sum(1 for value in full_counter.values() if value <= LONG_TAIL_THRESHOLD)
    sample_tail = sum(1 for value in sample_counter.values() if value <= LONG_TAIL_THRESHOLD)
    full_tail_mass = sum(value for value in full_counter.values() if value <= LONG_TAIL_THRESHOLD) / max(full_total, 1)
    sample_tail_mass = sum(value for value in sample_counter.values() if value <= LONG_TAIL_THRESHOLD) / max(sample_total, 1)
    return pd.DataFrame(
        [
            {
                "scope": "Full",
                "unique_labels": len(full_labels),
                "total_occurrences": full_total,
                f"long_tail_labels_<= {LONG_TAIL_THRESHOLD}": full_tail,
                "long_tail_mass_%": full_tail_mass * 100,
            },
            {
                "scope": sample_name,
                "unique_labels": len(sample_labels),
                "total_occurrences": sample_total,
                f"long_tail_labels_<= {LONG_TAIL_THRESHOLD}": sample_tail,
                "long_tail_mass_%": sample_tail_mass * 100,
            },
            {
                "scope": "Overlap",
                "unique_labels": len(shared_labels),
                "total_occurrences": np.nan,
                f"long_tail_labels_<= {LONG_TAIL_THRESHOLD}": np.nan,
                "long_tail_mass_%": np.nan,
            },
            {
                "scope": "Coverage",
                "labels_only_in_full": len(only_full),
                "labels_only_in_sample": len(only_sample),
                "coverage_full_%": len(shared_labels) / max(len(full_labels), 1) * 100,
                "coverage_sample_%": len(shared_labels) / max(len(sample_labels), 1) * 100,
            },
        ]
    )


def missing_labels_table(full_counter: Counter, sample_counter: Counter, top_n: int = TOP_N) -> pd.DataFrame:
    rows = [(label, count) for label, count in full_counter.items() if label not in sample_counter]
    rows.sort(key=lambda item: item[1], reverse=True)
    total_full = sum(full_counter.values())
    table = pd.DataFrame(rows[:top_n], columns=["label", "full_count"])
    if not table.empty:
        table["full_share_%"] = table["full_count"] / max(total_full, 1) * 100
    return table


def top_labels_table(counter: Counter, top_n: int = TOP_N) -> pd.DataFrame:
    total = sum(counter.values())
    rows = [(label, count, count / max(total, 1) * 100) for label, count in counter.most_common(top_n)]
    return pd.DataFrame(rows, columns=["label", "count", "share_%"])


def plot_rank_distribution(full_counter: Counter, sample_counter: Counter, title: str, sample_label: str) -> None:
    full_values = sorted_frequencies(full_counter)
    sample_values = sorted_frequencies(sample_counter)
    max_len = max(len(full_values), len(sample_values))
    if max_len == 0:
        print(f"[Skip] No labels available for {title}")
        return

    x = np.arange(1, max_len + 1)
    full_plot = np.full(max_len, np.nan)
    sample_plot = np.full(max_len, np.nan)
    full_plot[: len(full_values)] = full_values
    sample_plot[: len(sample_values)] = sample_values

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(x, full_plot, color="#1f77b4", linewidth=2.1, label="Full")
    ax.plot(x, sample_plot, color="#ff7f0e", linewidth=2.1, label=sample_label)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Label rank (log scale)")
    ax.set_ylabel("Relative frequency (log scale)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, which="both", alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_top_labels(full_counter: Counter, sample_counter: Counter, title: str, sample_label: str, top_n: int = TOP_N) -> None:
    full_table = top_labels_table(full_counter, top_n=top_n)
    sample_table = top_labels_table(sample_counter, top_n=top_n)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=False)
    for ax, table, subtitle, color in [
        (axes[0], full_table, "Full", "#1f77b4"),
        (axes[1], sample_table, sample_label, "#ff7f0e"),
    ]:
        if table.empty:
            ax.set_axis_off()
            continue
        labels = table["label"].iloc[::-1].tolist()
        shares = table["share_%"].iloc[::-1].to_numpy()
        counts = table["count"].iloc[::-1].to_numpy()
        ax.barh(labels, shares, color=color, alpha=0.85)
        for y, share, count in zip(range(len(labels)), shares, counts):
            ax.text(share + max(shares) * 0.01, y, f"{count:,}", va="center", fontsize=8)
        ax.set_title(subtitle)
        ax.set_xlabel("Share of all label occurrences (%)")
        ax.grid(axis="x", alpha=0.2)
    fig.suptitle(title, y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()


def summarize_bbox_arrays(name: str, arrays: Dict[str, List[float]]) -> pd.DataFrame:
    rows = []
    for key in ["width_px", "height_px", "area_px", "aspect_ratio", "norm_width", "norm_height", "area_ratio", "log2_aspect_ratio"]:
        values = np.asarray(arrays.get(key, []), dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            rows.append({"scope": name, "metric": key, "count": 0, "median": np.nan, "p90": np.nan, "p99": np.nan})
            continue
        rows.append(
            {
                "scope": name,
                "metric": key,
                "count": int(values.size),
                "median": float(np.median(values)),
                "p90": float(np.percentile(values, 90)),
                "p99": float(np.percentile(values, 99)),
            }
        )
    return pd.DataFrame(rows)


def plot_bbox_heatmaps(full_arrays: Dict[str, List[float]], sample_arrays: Dict[str, List[float]], title: str) -> None:
    pairs = [
        ("norm_width", "norm_height", "Normalized width vs height", (0, 1), (0, 1)),
        ("area_ratio", "log2_aspect_ratio", "Area ratio vs log2(aspect ratio)", (0, 1), (-4, 4)),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(18, 14), constrained_layout=True)
    for row_idx, (x_key, y_key, subtitle, xlim, ylim) in enumerate(pairs):
        for col_idx, (arrays, scope) in enumerate([(full_arrays, "Full"), (sample_arrays, "Sample 10k")]):
            ax = axes[row_idx, col_idx]
            x = np.asarray(arrays.get(x_key, []), dtype=float)
            y = np.asarray(arrays.get(y_key, []), dtype=float)
            mask = np.isfinite(x) & np.isfinite(y)
            x = x[mask]
            y = y[mask]
            if x.size == 0:
                ax.set_axis_off()
                continue
            hist = ax.hist2d(
                x,
                y,
                bins=HEATMAP_BINS,
                range=[xlim, ylim],
                norm=LogNorm(),
                cmap="magma",
            )
            ax.set_title(f"{scope} - {subtitle}")
            ax.set_xlabel(x_key.replace("_", " "))
            ax.set_ylabel(y_key.replace("_", " "))
            fig.colorbar(hist[3], ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(title, fontsize=18)
    plt.show()


def collect_object_stats(objects_data, image_ids: Optional[set[int]], image_sizes: Dict[int, Tuple[float, float]]):
    counter = Counter()
    arrays: Dict[str, List[float]] = {key: [] for key in ["width_px", "height_px", "area_px", "aspect_ratio", "norm_width", "norm_height", "area_ratio", "log2_aspect_ratio"]}
    meta = {
        "images": 0,
        "objects": 0,
        "empty_object_lists": 0,
        "invalid_bboxes": 0,
        "missing_image_size": 0,
        "empty_names": 0,
        "duplicate_object_ids": 0,
    }

    for image_record in objects_data:
        image_id = int(image_record.get("image_id"))
        if image_ids is not None and image_id not in image_ids:
            continue

        meta["images"] += 1
        objects = image_record.get("objects", [])
        if not objects:
            meta["empty_object_lists"] += 1

        seen_ids = set()
        img_w, img_h = image_sizes.get(image_id, (np.nan, np.nan))
        has_img_size = np.isfinite(img_w) and np.isfinite(img_h) and img_w > 0 and img_h > 0

        for obj in objects:
            object_id = obj.get("object_id")
            if object_id in seen_ids:
                meta["duplicate_object_ids"] += 1
            else:
                seen_ids.add(object_id)

            name = _normalize_object_name(str((obj.get("names") or [""])[0]).lower().strip())
            if not name:
                meta["empty_names"] += 1
                continue

            w = float(obj.get("w", 0) or 0)
            h = float(obj.get("h", 0) or 0)
            if w <= 0 or h <= 0:
                meta["invalid_bboxes"] += 1
                continue

            counter[name] += 1
            meta["objects"] += 1

            arrays["width_px"].append(w)
            arrays["height_px"].append(h)
            arrays["area_px"].append(w * h)
            arrays["aspect_ratio"].append(w / h)
            arrays["log2_aspect_ratio"].append(float(np.clip(np.log2(w / h), -4, 4)))

            if has_img_size:
                norm_w = min(w / img_w, 1.0)
                norm_h = min(h / img_h, 1.0)
                area_ratio = min((w * h) / (img_w * img_h), 1.0)
                arrays["norm_width"].append(norm_w)
                arrays["norm_height"].append(norm_h)
                arrays["area_ratio"].append(area_ratio)
            else:
                meta["missing_image_size"] += 1
                arrays["norm_width"].append(np.nan)
                arrays["norm_height"].append(np.nan)
                arrays["area_ratio"].append(np.nan)

    return counter, arrays, meta


def collect_attribute_stats(attributes_data, image_ids: Optional[set[int]]):
    counter = Counter()
    meta = {
        "images": 0,
        "attribute_objects": 0,
        "empty_attribute_lists": 0,
        "empty_attribute_tokens": 0,
        "duplicate_object_ids": 0,
    }

    for image_record in attributes_data:
        image_id = int(image_record.get("image_id"))
        if image_ids is not None and image_id not in image_ids:
            continue

        meta["images"] += 1
        attribute_objects = image_record.get("attributes", [])
        if not attribute_objects:
            meta["empty_attribute_lists"] += 1

        seen_ids = set()
        for attribute_record in attribute_objects:
            object_id = attribute_record.get("object_id")
            if object_id in seen_ids:
                meta["duplicate_object_ids"] += 1
            else:
                seen_ids.add(object_id)

            tokens = [str(token).lower().strip() for token in attribute_record.get("attributes", []) if str(token).strip()]
            if not tokens:
                meta["empty_attribute_tokens"] += 1
                continue

            counter.update(tokens)
            meta["attribute_objects"] += 1

    return counter, meta


def collect_relation_stats(relations_data, image_ids: Optional[set[int]], image_sizes: Dict[int, Tuple[float, float]]):
    counter = Counter()
    arrays: Dict[str, List[float]] = {key: [] for key in ["width_px", "height_px", "area_px", "aspect_ratio", "norm_width", "norm_height", "area_ratio", "log2_aspect_ratio"]}
    meta = {
        "images": 0,
        "relations": 0,
        "empty_relation_lists": 0,
        "empty_predicates": 0,
        "invalid_union_bboxes": 0,
        "missing_image_size": 0,
    }

    for image_record in relations_data:
        image_id = int(image_record.get("image_id"))
        if image_ids is not None and image_id not in image_ids:
            continue

        meta["images"] += 1
        relation_records = image_record.get("relationships", [])
        if not relation_records:
            meta["empty_relation_lists"] += 1

        img_w, img_h = image_sizes.get(image_id, (np.nan, np.nan))
        has_img_size = np.isfinite(img_w) and np.isfinite(img_h) and img_w > 0 and img_h > 0

        for relation in relation_records:
            predicate = _normalize_relation_name(str(relation.get("predicate", "")).lower().strip())
            if not predicate:
                meta["empty_predicates"] += 1
                continue

            subject = relation.get("subject", {})
            object_ = relation.get("object", {})
            x1 = min(float(subject.get("x", 0) or 0), float(object_.get("x", 0) or 0))
            y1 = min(float(subject.get("y", 0) or 0), float(object_.get("y", 0) or 0))
            x2 = max(float(subject.get("x", 0) or 0) + float(subject.get("w", 0) or 0), float(object_.get("x", 0) or 0) + float(object_.get("w", 0) or 0))
            y2 = max(float(subject.get("y", 0) or 0) + float(subject.get("h", 0) or 0), float(object_.get("y", 0) or 0) + float(object_.get("h", 0) or 0))
            w = x2 - x1
            h = y2 - y1

            if w <= 0 or h <= 0:
                meta["invalid_union_bboxes"] += 1
                continue

            counter[predicate] += 1
            meta["relations"] += 1

            arrays["width_px"].append(w)
            arrays["height_px"].append(h)
            arrays["area_px"].append(w * h)
            arrays["aspect_ratio"].append(w / h)
            arrays["log2_aspect_ratio"].append(float(np.clip(np.log2(w / h), -4, 4)))

            if has_img_size:
                norm_w = min(w / img_w, 1.0)
                norm_h = min(h / img_h, 1.0)
                area_ratio = min((w * h) / (img_w * img_h), 1.0)
                arrays["norm_width"].append(norm_w)
                arrays["norm_height"].append(norm_h)
                arrays["area_ratio"].append(area_ratio)
            else:
                meta["missing_image_size"] += 1
                arrays["norm_width"].append(np.nan)
                arrays["norm_height"].append(np.nan)
                arrays["area_ratio"].append(np.nan)

    return counter, arrays, meta


def plot_label_section(label_title: str, full_counter: Counter, sample_counter: Counter, sample_label: str) -> None:
    print(f"\n### {label_title}")
    display(label_coverage_summary(full_counter, sample_counter, sample_name=sample_label))
    print(f"Missing labels in {sample_label} (top {TOP_N} by full frequency)")
    display(missing_labels_table(full_counter, sample_counter, top_n=TOP_N))
    plot_rank_distribution(full_counter, sample_counter, f"{label_title}: full vs {sample_label} rank-frequency", sample_label)
    plot_top_labels(full_counter, sample_counter, f"{label_title}: top {TOP_N} labels", sample_label, top_n=TOP_N)


def plot_bbox_section(title: str, full_arrays: Dict[str, List[float]], sample_arrays: Dict[str, List[float]]) -> None:
    display(pd.concat([summarize_bbox_arrays("Full", full_arrays), summarize_bbox_arrays("Sample 10k", sample_arrays)], ignore_index=True))
    plot_bbox_heatmaps(full_arrays, sample_arrays, title)


print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj
Raw dir: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\raw
Processed dir: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed


In [ ]:
image_sizes = load_image_size_map(RAW_DIR / "image_data.json")
object_sample_ids, object_split_sizes = load_processed_image_ids(OBJECT_PROCESSED_DIR)
attribute_sample_ids, attribute_split_sizes = load_processed_image_ids(ATTRIBUTE_PROCESSED_DIR)
relation_sample_ids, relation_split_sizes = load_processed_image_ids(RELATION_PROCESSED_DIR)

sample_overview = pd.DataFrame(
    [
        {
            "task": "object",
            "train_samples": object_split_sizes.get("train", 0),
            "val_samples": object_split_sizes.get("val", 0),
            "test_samples": object_split_sizes.get("test", 0),
            "unique_images": len(object_sample_ids),
        },
        {
            "task": "attribute",
            "train_samples": attribute_split_sizes.get("train", 0),
            "val_samples": attribute_split_sizes.get("val", 0),
            "test_samples": attribute_split_sizes.get("test", 0),
            "unique_images": len(attribute_sample_ids),
        },
        {
            "task": "relation",
            "train_samples": relation_split_sizes.get("train", 0),
            "val_samples": relation_split_sizes.get("val", 0),
            "test_samples": relation_split_sizes.get("test", 0),
            "unique_images": len(relation_sample_ids),
        },
    ]
)
print("=== Sample overview ===")
display(sample_overview)

print("=== Task 1: Object labels ===")
objects_data = load_json(RAW_DIR / "objects.json")
object_full_counter, object_full_arrays, object_full_meta = collect_object_stats(objects_data, None, image_sizes)
object_sample_counter, object_sample_arrays, object_sample_meta = collect_object_stats(objects_data, object_sample_ids, image_sizes)
display(pd.DataFrame([object_full_meta, object_sample_meta], index=["Full", "Sample 10k"]))
plot_label_section("Object labels", object_full_counter, object_sample_counter, "Sample 10k")
plot_bbox_section("Task 1 object bbox distributions", object_full_arrays, object_sample_arrays)
del objects_data

gc.collect()

print("=== Task 1: Attribute labels ===")
attributes_data = load_json(RAW_DIR / "attributes.json")
attribute_full_counter, _, attribute_full_meta = collect_attribute_stats(attributes_data, None)
attribute_sample_counter, _, attribute_sample_meta = collect_attribute_stats(attributes_data, attribute_sample_ids)
display(pd.DataFrame([attribute_full_meta, attribute_sample_meta], index=["Full", "Sample 10k"]))
plot_label_section("Attribute labels", attribute_full_counter, attribute_sample_counter, "Sample 10k")
del attributes_data

gc.collect()

print("=== Task 2: Relation labels ===")
relations_data = load_json(RAW_DIR / "relationships.json")
relation_full_counter, relation_full_arrays, relation_full_meta = collect_relation_stats(relations_data, None, image_sizes)
relation_sample_counter, relation_sample_arrays, relation_sample_meta = collect_relation_stats(relations_data, relation_sample_ids, image_sizes)
display(pd.DataFrame([relation_full_meta, relation_sample_meta], index=["Full", "Sample 10k"]))
plot_label_section("Relation labels", relation_full_counter, relation_sample_counter, "Sample 10k")
plot_bbox_section("Task 2 relation union bbox distributions", relation_full_arrays, relation_sample_arrays)
del relations_data

gc.collect()

print("EDA complete.")

## Cách đọc kết quả

- Biểu đồ rank-frequency dùng để nhìn độ lệch long-tail giữa full và sample 10k.
- Biểu đồ top labels dùng share (%) nên full và sample có thể đặt cạnh nhau trực tiếp.
- Heatmap bbox dùng bin nhỏ để thấy rõ vùng tập trung của width/height và area/aspect ratio.
- Phần phân bố split train/val/test đã được bỏ khỏi notebook này theo yêu cầu.